<a href="https://colab.research.google.com/github/dongqianqi-888/data_management--case_study--spark_in_iris_dataset/blob/main/dm_spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install Java and Spark
!apt-get update
!apt-get install -y openjdk-8-jdk-headless
!wget https://archive.apache.org/dist/spark/spark-3.3.0/spark-3.3.0-bin-hadoop3.tgz
!tar -xzf spark-3.3.0-bin-hadoop3.tgz

# Set environment variables
!echo "export SPARK_HOME=\$PWD/spark-3.3.0-bin-hadoop3" >> ~/.bashrc
!echo "export PATH=\$PATH:\$SPARK_HOME/bin" >> ~/.bashrc
!source ~/.bashrc

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,070 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,945 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 

In [2]:
# Start Spark
!$SPARK_HOME/sbin/start-master.sh
!$SPARK_HOME/sbin/start-worker.sh \$SPARK_MASTER_URL

/bin/bash: line 1: /sbin/start-master.sh: No such file or directory
/bin/bash: line 1: /sbin/start-worker.sh: No such file or directory


In [3]:
# Check Spark log
!tail -n 20 \$SPARK_HOME/logs/spark-\*-org.apache.spark.deploy.master.Master-\*.log

tail: cannot open '$SPARK_HOME/logs/spark-*-org.apache.spark.deploy.master.Master-*.log' for reading: No such file or directory


In [10]:
# Import necessary libraries
import time
from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder.appName("IrisExample").getOrCreate()

# read the iris
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
pdf = pd.read_csv(url, header=None)
iris_df = spark.createDataFrame(pdf).toDF(
    "sepal_length", "sepal_width", "petal_length", "petal_width", "species"
)

print("---- iris ----")
iris_df.show(5)
print("---- iris ----")
iris_df.printSchema()

# cleaning
print("---- iris_clean ----")
missing_counts = iris_df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in iris_df.columns
])
print("The number of outliers:",missing_counts.show())
pdf = iris_df.toPandas()
numeric_df = pdf.select_dtypes(include=[np.number])
z_scores = np.abs(zscore(numeric_df))
outliers = (z_scores > 3).any(axis=1)

print("The number of outliers:", outliers.sum())

label_indexer = StringIndexer(inputCol="species", outputCol="label")
assembler = VectorAssembler(
    inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
    outputCol="features"
)

pipeline = Pipeline(stages=[label_indexer, assembler])
processed_data = pipeline.fit(iris_df).transform(iris_df)

processed_data.select("features", "label").show(5)

# trainset and testset
train_data, test_data = processed_data.randomSplit([0.7, 0.3], seed=123)
print(f"size of training:{train_data.count()}，size of testing:{test_data.count()}")

# ---------------- Define Models ----------------
lr = LogisticRegression(labelCol="label", featuresCol="features")
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features", seed=123)
rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=123)

# ---------------- Define Evaluator ----------------
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

# ---------------- Start Total Timing ----------------
start_total = time.time()

# ---------------- Logistic Regression with CrossValidation ----------------
start_train_lr = time.time()

lr_paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.maxIter, [10, 20, 50]) \
    .build()

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_paramGrid,
    evaluator=evaluator,
    numFolds=3,
    seed=357
)

try:
    lr_cv_model = lr_cv.fit(train_data)
    lr_best_model = lr_cv_model.bestModel
except Exception as e:
    print("Logistic Regression training failed")
    lr_best_model = None

train_time_lr = time.time() - start_train_lr

# ---------------- Decision Tree with CrossValidation ----------------
start_train_dt = time.time()

dt_paramGrid = ParamGridBuilder() \
    .addGrid(dt.maxDepth, [3, 5, 7, 10]) \
    .addGrid(dt.minInstancesPerNode, [1, 2, 4]) \
    .build()

dt_cv = CrossValidator(
    estimator=dt,
    estimatorParamMaps=dt_paramGrid,
    evaluator=evaluator,
    numFolds=3,
    seed=357
)

dt_cv_model = dt_cv.fit(train_data)
dt_best_model = dt_cv_model.bestModel

train_time_dt = time.time() - start_train_dt

# ---------------- Random Forest with CrossValidation ----------------
start_train_rf = time.time()

rf_paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20, 50]) \
    .addGrid(rf.maxDepth, [3, 5, 7]) \
    .build()

rf_cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=rf_paramGrid,
    evaluator=evaluator,
    numFolds=3,
    seed=357
)

rf_cv_model = rf_cv.fit(train_data)
rf_best_model = rf_cv_model.bestModel

train_time_rf = time.time() - start_train_rf

# ---------------- Define Evaluation Function ----------------
def evaluate_model(model, name, data):
    if model is None:
        return {"model": name, "accuracy": 0, "precision": 0, "recall": 0, "F1": 0}
    preds = model.transform(data)
    return {
        "model": name,
        "accuracy": round(evaluator.evaluate(preds, {evaluator.metricName: "accuracy"}), 4),
        "precision": round(evaluator.evaluate(preds, {evaluator.metricName: "weightedPrecision"}), 4),
        "recall": round(evaluator.evaluate(preds, {evaluator.metricName: "weightedRecall"}), 4),
        "F1": round(evaluator.evaluate(preds, {evaluator.metricName: "f1"}), 4)
    }

# ---------------- Evaluate on Training Set ----------------
print("\n---- Best Model in Training Set ----")
results = [
    evaluate_model(lr_best_model, "Logistic Regression", train_data),
    evaluate_model(dt_best_model, "Decision Tree", train_data),
    evaluate_model(rf_best_model, "Random Forest", train_data)
]

for r in results:
    print(r)

best_res = max(results, key=lambda x: x["F1"])
print(f"\nBest model on training set: {best_res['model']}")

# ---------------- Evaluate on Test Set ----------------
print("\n---- Model Performance on Test Set ----")
test_results = [
    evaluate_model(lr_best_model, "Logistic Regression", test_data),
    evaluate_model(dt_best_model, "Decision Tree", test_data),
    evaluate_model(rf_best_model, "Random Forest", test_data)
]

for r in test_results:
    print(r)

best_res_test = max(test_results, key=lambda x: x["F1"])
print(f"\nBest model on test set: {best_res_test['model']}")

# ---------------- Calculate Total Time ----------------
total_time = time.time() - start_total

# ---------------- Print Time Report ----------------
print("\n" + "="*60)
print("              MODEL TRAINING TIME REPORT              ")
print("="*60)
print(f"Logistic Regression Training Time: {train_time_lr:.2f} sec")
print(f"Decision Tree Training Time:       {train_time_dt:.2f} sec")
print(f"Random Forest Training Time:       {train_time_rf:.2f} sec")
print(f"Total Running Time:                {total_time:.2f} sec")
print("="*60)

---- iris ----
+------------+-----------+------------+-----------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|    species|
+------------+-----------+------------+-----------+-----------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|
+------------+-----------+------------+-----------+-----------+
only showing top 5 rows
---- iris ----
root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)

---- iris_clean ----
+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+---------

In [11]:
# Import necessary libraries
import time
from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder.appName("IrisExample").getOrCreate()

# read the iris
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
pdf = pd.read_csv(url, header=None)
iris_df = spark.createDataFrame(pdf).toDF(
    "sepal_length", "sepal_width", "petal_length", "petal_width", "species"
)

print("---- iris ----")
iris_df.show(5)
print("---- iris ----")
iris_df.printSchema()

# cleaning
print("---- iris_clean ----")
missing_counts = iris_df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in iris_df.columns
])
print("The number of outliers:",missing_counts.show())
pdf = iris_df.toPandas()
numeric_df = pdf.select_dtypes(include=[np.number])
z_scores = np.abs(zscore(numeric_df))
outliers = (z_scores > 3).any(axis=1)

print("The number of outliers:", outliers.sum())

label_indexer = StringIndexer(inputCol="species", outputCol="label")
assembler = VectorAssembler(
    inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
    outputCol="features"
)

pipeline = Pipeline(stages=[label_indexer, assembler])
processed_data = pipeline.fit(iris_df).transform(iris_df)

processed_data.select("features", "label").show(5)

# trainset and testset
train_data, test_data = processed_data.randomSplit([0.7, 0.3], seed=123)
print(f"size of training:{train_data.count()}，size of testing:{test_data.count()}")

# ---------------- Define Models ----------------
lr = LogisticRegression(labelCol="label", featuresCol="features")
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features", seed=123)
rf = RandomForestClassifier(labelCol="label", featuresCol="features", seed=123)

# ---------------- Define Evaluator ----------------
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

# ---------------- Start Total Timing ----------------
start_total = time.time()

# ---------------- Logistic Regression with CrossValidation ----------------
start_train_lr = time.time()

lr_paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.maxIter, [10, 20, 50]) \
    .build()

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_paramGrid,
    evaluator=evaluator,
    numFolds=10,
    seed=357
)

try:
    lr_cv_model = lr_cv.fit(train_data)
    lr_best_model = lr_cv_model.bestModel
except Exception as e:
    print("Logistic Regression training failed")
    lr_best_model = None

train_time_lr = time.time() - start_train_lr

# ---------------- Decision Tree with CrossValidation ----------------
start_train_dt = time.time()

dt_paramGrid = ParamGridBuilder() \
    .addGrid(dt.maxDepth, [3, 5, 7, 10]) \
    .addGrid(dt.minInstancesPerNode, [1, 2, 4]) \
    .build()

dt_cv = CrossValidator(
    estimator=dt,
    estimatorParamMaps=dt_paramGrid,
    evaluator=evaluator,
    numFolds=10,
    seed=357
)

dt_cv_model = dt_cv.fit(train_data)
dt_best_model = dt_cv_model.bestModel

train_time_dt = time.time() - start_train_dt

# ---------------- Random Forest with CrossValidation ----------------
start_train_rf = time.time()

rf_paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20, 50]) \
    .addGrid(rf.maxDepth, [3, 5, 7]) \
    .build()

rf_cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=rf_paramGrid,
    evaluator=evaluator,
    numFolds=10,
    seed=357
)

rf_cv_model = rf_cv.fit(train_data)
rf_best_model = rf_cv_model.bestModel

train_time_rf = time.time() - start_train_rf

# ---------------- Define Evaluation Function ----------------
def evaluate_model(model, name, data):
    if model is None:
        return {"model": name, "accuracy": 0, "precision": 0, "recall": 0, "F1": 0}
    preds = model.transform(data)
    return {
        "model": name,
        "accuracy": round(evaluator.evaluate(preds, {evaluator.metricName: "accuracy"}), 4),
        "precision": round(evaluator.evaluate(preds, {evaluator.metricName: "weightedPrecision"}), 4),
        "recall": round(evaluator.evaluate(preds, {evaluator.metricName: "weightedRecall"}), 4),
        "F1": round(evaluator.evaluate(preds, {evaluator.metricName: "f1"}), 4)
    }

# ---------------- Evaluate on Training Set ----------------
print("\n---- Best Model in Training Set ----")
results = [
    evaluate_model(lr_best_model, "Logistic Regression", train_data),
    evaluate_model(dt_best_model, "Decision Tree", train_data),
    evaluate_model(rf_best_model, "Random Forest", train_data)
]

for r in results:
    print(r)

best_res = max(results, key=lambda x: x["F1"])
print(f"\nBest model on training set: {best_res['model']}")

# ---------------- Evaluate on Test Set ----------------
print("\n---- Model Performance on Test Set ----")
test_results = [
    evaluate_model(lr_best_model, "Logistic Regression", test_data),
    evaluate_model(dt_best_model, "Decision Tree", test_data),
    evaluate_model(rf_best_model, "Random Forest", test_data)
]

for r in test_results:
    print(r)

best_res_test = max(test_results, key=lambda x: x["F1"])
print(f"\nBest model on test set: {best_res_test['model']}")

# ---------------- Calculate Total Time ----------------
total_time = time.time() - start_total

# ---------------- Print Time Report ----------------
print("\n" + "="*60)
print("              MODEL TRAINING TIME REPORT              ")
print("="*60)
print(f"Logistic Regression Training Time: {train_time_lr:.2f} sec")
print(f"Decision Tree Training Time:       {train_time_dt:.2f} sec")
print(f"Random Forest Training Time:       {train_time_rf:.2f} sec")
print(f"Total Running Time:                {total_time:.2f} sec")
print("="*60)

---- iris ----
+------------+-----------+------------+-----------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|    species|
+------------+-----------+------------+-----------+-----------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|
+------------+-----------+------------+-----------+-----------+
only showing top 5 rows
---- iris ----
root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)

---- iris_clean ----
+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+---------